In [1]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ================= 配置区 =================
RAW_KG_PATH = "kg.csv"

def batch_inspect_ontology():
    print("==================================================")
    print("🛠️ 步骤 1: 启动 PrimeKG 本体对齐批量检修工具...")
    
    try:
        # low_memory=False 防止大文件读取警告
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
        print(f"   ✅ 成功加载 {RAW_KG_PATH}。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {RAW_KG_PATH}，请检查路径。")
        return

    print("\n==================================================")
    print("🧹 步骤 2: 划定安全检索范围...")
    # 过滤出只属于疾病和表型的子集，加快检索速度，最重要的是防止搜出一堆同名的基因或通路噪音
    target_types = ['disease', 'effect/phenotype']
    search_df = df[df['x_type'].isin(target_types)]
    print(f"   已将检索范围锁定在 {len(search_df)} 条 'disease' 和 'effect/phenotype' 关系中。")
    
    # 针对刚才失败的特征定义搜索策略 (ADNI特征: [模糊搜索词根列表])
    search_strategy = {
        "npiq_DEPD / gds (抑郁)": ["depress", "major depressive"],
        "npiq_NITE (睡眠障碍)": ["sleep", "insomnia"],
        "npiq_APP (饮食/食欲异常)": ["eat", "appetite", "feed"],
        "trail / faq (执行功能/认知)": ["executive", "cognition", "cognitive decline"],
        "digit (注意力)": ["attention", "deficit"],
        "his_CVHATT (心血管/心梗)": ["myocardial", "heart attack", "cardiac"],
        "his_CBSTROKE (中风/脑卒中)": ["stroke", "cerebral infarction", "cerebrovascular"],
        "his_HYPERTEN (高血压)": ["hypertens"],
        "his_PSYCDIS (精神疾病)": ["mental", "psychiatric", "psychosis"]
    }

    print("\n==================================================")
    print("🔍 步骤 3: 开始按医学词根进行批量模糊检索...")
    
    for category, keywords in search_strategy.items():
        print(f"\n▶ 正在检索【{category}】...")
        print(f"   使用的词根: {keywords}")
        
        # 构建正则表达式，多个词根用 | 连接，例如 'sleep|insomnia'
        pattern = '|'.join(keywords)
        
        # 在 x_name 列进行正则匹配，case=False 表示忽略大小写
        mask = search_df['x_name'].str.contains(pattern, case=False, na=False)
        results = search_df[mask]['x_name'].unique()
        
        if len(results) > 0:
            print(f"   ✅ 找到 {len(results)} 个候选节点，前 10 个最精准的候选如下：")
            # 简单排序：把长度较短的（通常是更基础的本体标准词）放前面
            sorted_results = sorted(results, key=len)
            for i, res in enumerate(sorted_results[:10]):
                print(f"      [{i+1}] {res}")
        else:
            print(f"   ❌ 使用这些词根未能找到任何节点，PrimeKG 中可能使用了完全不同的医学术语。")

    print("\n==================================================")
    print("💡 步骤 4: 检修结果输出完毕。")
    print("   请从上述候选项中挑选最贴切的准确学名（请原样复制，区分大小写），")
    print("   将它们替换回第一个脚本的 adni_mapping 字典中。")
    print("==================================================")

if __name__ == "__main__":
    batch_inspect_ontology()

🛠️ 步骤 1: 启动 PrimeKG 本体对齐批量检修工具...
   ✅ 成功加载 kg.csv。

🧹 步骤 2: 划定安全检索范围...
   已将检索范围锁定在 598340 条 'disease' 和 'effect/phenotype' 关系中。

🔍 步骤 3: 开始按医学词根进行批量模糊检索...

▶ 正在检索【npiq_DEPD / gds (抑郁)】...
   使用的词根: ['depress', 'major depressive']
   ✅ 找到 24 个候选节点，前 10 个最精准的候选如下：
      [1] Depressivity
      [2] Depressed glabella
      [3] Metopic depression
      [4] unipolar depression
      [5] Depressed nasal tip
      [6] neurotic depression
      [7] endogenous depression
      [8] postpartum depression
      [9] Depressed nasal ridge
      [10] ST segment depression

▶ 正在检索【npiq_NITE (睡眠障碍)】...
   使用的词根: ['sleep', 'insomnia']
   ✅ 找到 48 个候选节点，前 10 个最精准的候选如下：
      [1] Sleep terror
      [2] Sleep myoclonus
      [3] Sleep paralysis
      [4] Sleep disturbance
      [5] Terminal insomnia
      [6] insomnia (disease)
      [7] Sleep-interrupting
      [8] sleep-wake disorder
      [9] Central sleep apnea
      [10] complex sleep apnea

▶ 正在检索【npiq_APP (饮食/食欲异常)】...
   使用的词根: ['eat', 'appetite'